In [1]:
from clickhouse_driver import Client

client = Client(host='clickhouse', port=9000)

result = client.execute('SELECT version()')
print(f'ClickHouse version: {result[0][0]}')

ClickHouse version: 23.8.16.16


In [2]:
result = client.execute("""
    SELECT name, engine
    FROM system.tables
    WHERE database = 'default'
    ORDER BY name
""")

for name, engine in result:
    print(name, engine)

currency_rates ReplacingMergeTree
dict_account_level Dictionary
dict_account_level_source MergeTree
dict_country Dictionary
dict_country_source MergeTree
dict_merchant_category Dictionary
dict_merchant_category_source MergeTree
dm_analytics_daily ReplacingMergeTree
dm_antifraud_daily ReplacingMergeTree
dm_daily_limits ReplacingMergeTree
dm_monthly_limits ReplacingMergeTree
kafka_transactions_invalid Kafka
kafka_transactions_valid Kafka
mv_transactions_invalid MaterializedView
mv_transactions_valid MaterializedView
transactions_invalid ReplacingMergeTree
transactions_valid ReplacingMergeTree


In [3]:
import pandas as pd
pd.DataFrame(
    client.execute("select * from currency_rates FINAL"), 
    columns=('dt', 'currency', 'rate_to_rub', 'nominal', 'loaded_at')
).sort_values(by='dt').tail(10)

/tmp/ipykernel_827/2380232174.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


,dt,currency,rate_to_rub,nominal,loaded_at
11266,2026-04-19,USD,76.0535,1,2026-06-04 11:49:28
11270,2026-04-20,USD,76.0535,1,2026-06-04 11:49:28
11267,2026-04-20,EUR,89.6256,1,2026-06-04 11:49:28
11268,2026-04-20,GBP,102.8395,1,2026-06-04 11:49:28
11269,2026-04-20,RUB,1,1,2026-06-04 11:49:28
11274,2026-04-22,USD,74.5897,1,2026-06-04 11:49:28
11273,2026-04-22,RUB,1,1,2026-06-04 11:49:28
11271,2026-04-22,EUR,87.7659,1,2026-06-04 11:49:28
11272,2026-04-22,GBP,100.8751,1,2026-06-04 11:49:28
11275,2026-12-01,RUB,1,1,2026-06-04 11:49:28


In [4]:
merchant_categories = [
    ('other', 'Other', 9, 0),
    ('personal_transfer', 'Personal Transfer', 8, 1),
    ('ATM', 'ATM Withdrawal', 7, 0),
    ('utilities', 'Utilities & Bills', 4, 1),
    ('entertainment', 'Entertainment', 4, 1),
    ('transport', 'Transport', 2, 0),
    ('food', 'Food & Dining', 2, 0),
    ('retail', 'Retail Shopping', 2, 0),
]

client.execute("TRUNCATE TABLE dict_merchant_category_source")

client.execute(
    'INSERT INTO dict_merchant_category_source '
    '(category_code, category_name, risk_score, is_online) VALUES',
    merchant_categories
)

8

In [5]:
result = client.execute(
    'SELECT category_code, category_name, risk_score, is_online '
    'FROM dict_merchant_category_source '
    'ORDER BY risk_score DESC'
)

pd.DataFrame(result)

,0,1,2,3
0,other,Other,9,0
1,personal_transfer,Personal Transfer,8,1
2,ATM,ATM Withdrawal,7,0
3,entertainment,Entertainment,4,1
4,utilities,Utilities & Bills,4,1
5,food,Food & Dining,2,0
6,retail,Retail Shopping,2,0
7,transport,Transport,2,0


In [6]:
countries = [
    ('US', 'United States',  'Americas', 1),
    ('GB', 'United Kingdom', 'Europe',   1),
    ('FR', 'France',         'Europe',   1),
    ('DE', 'Germany',        'Europe',   1),
    ('JP', 'Japan',          'Asia',     2),
    ('RU', 'Russia',         'Europe',   3),
]

client.execute('TRUNCATE TABLE dict_country_source')
client.execute(
    'INSERT INTO dict_country_source '
    '(country_code, country_name, region, risk_level) VALUES',
    countries
)

6

In [7]:
result = client.execute(
    'SELECT country_code, country_name, region, risk_level '
    'FROM dict_country_source '
    'ORDER BY risk_level, country_code'
)

pd.DataFrame(result)

,0,1,2,3
0,DE,Germany,Europe,1
1,FR,France,Europe,1
2,GB,United Kingdom,Europe,1
3,US,United States,Americas,1
4,JP,Japan,Asia,2
5,RU,Russia,Europe,3


In [8]:
from decimal import Decimal

account_levels = [
    ('standard', 'Standard', Decimal('10000.00'),  Decimal('50000.00')),
    ('premium',  'Premium',  Decimal('25000.00'),  Decimal('150000.00')),
    ('vip',      'VIP',      Decimal('100000.00'), Decimal('500000.00')),
]

client.execute('TRUNCATE TABLE dict_account_level_source')
client.execute(
    'INSERT INTO dict_account_level_source '
    '(level_code, level_name, daily_limit_rub, monthly_limit_rub) VALUES',
    account_levels
)

3

In [9]:
result = client.execute(
    'SELECT level_code, level_name, daily_limit_rub, monthly_limit_rub '
    'FROM dict_account_level_source '
    'ORDER BY daily_limit_rub'
)

pd.DataFrame(result)

,0,1,2,3
0,standard,Standard,10000,50000
1,premium,Premium,25000,150000
2,vip,VIP,100000,500000


In [10]:
client.execute("SYSTEM RELOAD DICTIONARY dict_account_level")
client.execute("SYSTEM RELOAD DICTIONARY dict_merchant_category")
client.execute("SYSTEM RELOAD DICTIONARY dict_country")

[]